# Task 3: LID-augmented MuRIL
Feeds Task 1's predicted language tags into MuRIL. Each subword gets its word's tag, embedded in 16 dimensions and concatenated to the subword's word embedding before the transformer layers. The input text is Dakshina-normalized, and the hyperparameters are those of Task 2.

- **Inputs:** `data/partition_b_*_processed.json` (from Task 1)
- **Outputs:** `results/task3_metrics.json`, `results/task3_predictions.json`
- **Runtime:** **T4 GPU** (Runtime → Change runtime type), about 9 minutes.

## Setup

In [ ]:
# Work in a folder on Google Drive so all notebooks share data/ and results/
import os
from google.colab import drive

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/NLP_Group17"
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU, then run again."
print("GPU:", torch.cuda.get_device_name(0))

## Configuration

In [ ]:
import os
import json
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, accuracy_score, classification_report
from tqdm import tqdm

SEED = 42
MODEL_NAME = "google/muril-base-cased"
DATA_DIR = "data"
RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

LABEL2ID = {"negative": 0, "neutral": 1, "positive": 2}
ID2LABEL = {0: "negative", 1: "neutral", 2: "positive"}
LID2ID = {"O": 0, "Eng": 1, "Hin": 2, "PAD": 3}

In [ ]:
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

## Dataset
Tokenizes the normalized words and gives every subword the predicted tag of its word. Special and padding tokens get a separate `PAD` tag.

In [ ]:
class LIDAugmentedTweetDataset(Dataset):
    def __init__(self, items, tokenizer, max_len=128):
        self.items = items
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        words = item["norm_tokens"]
        lid_tags = item["pred_lang_tags"]
        label = LABEL2ID[item["sentiment"]]

        enc = self.tokenizer(
            words,
            is_split_into_words=True,
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt"
        )
        word_ids = enc.word_ids(batch_index=0)
        subword_lids = []
        for wid in word_ids:
            if wid is None:
                subword_lids.append(LID2ID["PAD"])
            else:
                tag = lid_tags[wid] if wid < len(lid_tags) else "O"
                subword_lids.append(LID2ID.get(tag, 0))

        return {
            "id": item["id"],
            "text": item["norm_text"],
            "raw_text": item["raw_text"],
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "lid_ids": torch.tensor(subword_lids, dtype=torch.long),
            "labels": torch.tensor(label, dtype=torch.long)
        }

## Model
The 784-dimensional concatenation `[word embedding ; tag embedding]` is projected back to 768 dimensions with `[I | W_lid]`. The word block stays the identity and only `W_lid` (initialized to zero) is learned, so training starts from exactly the pretrained MuRIL. A fully learned 784×768 projection made training collapse to a single class (see the report).

In [ ]:
class LIDAugmentedMuRIL(nn.Module):
    """
    Concatenates a learned language-tag embedding (dim=16) to each subword's MuRIL word
    embedding before the transformer layers, then projects the 784-dim concatenation back to
    hidden_size (768):

        inputs_embeds = P [word_embed ; lid_embed],   P = [ I_768 | W_lid ]

    The word block of P is fixed at the identity and only W_lid (16 -> 768, zero-initialized)
    is learned, so at step 0 the model is exactly pretrained MuRIL. A fully learned 784x768
    projection is NOT used: AdamW moves every one of its ~600k weights by ~lr per step, and
    summed over 768 inputs that drift quickly outweighs MuRIL's small word embeddings
    (std ~0.07), wiping out the pretrained input space (observed: the model collapsed to
    predicting one class).
    """
    def __init__(self, model_name=MODEL_NAME, lid_dim=16, num_labels=3):
        super().__init__()
        self.muril = AutoModel.from_pretrained(model_name)
        hidden_size = self.muril.config.hidden_size

        self.lid_embedding = nn.Embedding(num_embeddings=4, embedding_dim=lid_dim, padding_idx=LID2ID["PAD"])
        self.lid_proj = nn.Linear(lid_dim, hidden_size, bias=False)  # W_lid
        nn.init.zeros_(self.lid_proj.weight)

        self.dropout = nn.Dropout(self.muril.config.hidden_dropout_prob)
        self.classifier = nn.Linear(hidden_size, num_labels)
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, input_ids, attention_mask, lid_ids, labels=None):
        word_embeds = self.muril.embeddings.word_embeddings(input_ids)
        lid_embeds = self.lid_embedding(lid_ids)

        # = [I | W_lid] @ concat(word_embeds, lid_embeds)
        inputs_embeds = word_embeds + self.lid_proj(lid_embeds)

        outputs = self.muril(inputs_embeds=inputs_embeds, attention_mask=attention_mask)
        pooled = outputs.pooler_output if outputs.pooler_output is not None else outputs.last_hidden_state[:, 0]
        logits = self.classifier(self.dropout(pooled))

        loss = self.loss_fn(logits, labels) if labels is not None else None
        return loss, logits

## Evaluation

In [ ]:
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels, records = [], [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            lid_ids = batch["lid_ids"].to(device)
            labels = batch["labels"].to(device)
            _, logits = model(input_ids=input_ids, attention_mask=attention_mask, lid_ids=lid_ids)
            preds = torch.argmax(logits, dim=-1)
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())
            for tid, rtxt, ntxt, y_true, y_pred in zip(
                batch["id"], batch["raw_text"], batch["text"], labels.cpu().numpy(), preds.cpu().numpy()
            ):
                records.append({
                    "id": tid,
                    "raw_text": rtxt,
                    "norm_text": ntxt,
                    "gold": ID2LABEL[int(y_true)],
                    "pred": ID2LABEL[int(y_pred)]
                })
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    acc = accuracy_score(all_labels, all_preds)
    return macro_f1, acc, all_labels, all_preds, records

## Training and testing
Same optimizer, schedule and best-epoch selection as Task 2.

In [ ]:
def main():
    set_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    with open(os.path.join(DATA_DIR, "partition_b_train_processed.json"), "r", encoding="utf-8") as f:
        train_data = json.load(f)
    with open(os.path.join(DATA_DIR, "partition_b_validation_processed.json"), "r", encoding="utf-8") as f:
        val_data = json.load(f)
    with open(os.path.join(DATA_DIR, "partition_b_test_processed.json"), "r", encoding="utf-8") as f:
        test_data = json.load(f)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = LIDAugmentedMuRIL(MODEL_NAME, lid_dim=16, num_labels=3).to(device)

    train_loader = DataLoader(LIDAugmentedTweetDataset(train_data, tokenizer, 128), batch_size=16, shuffle=True)
    val_loader = DataLoader(LIDAugmentedTweetDataset(val_data, tokenizer, 128), batch_size=32, shuffle=False)
    test_loader = DataLoader(LIDAugmentedTweetDataset(test_data, tokenizer, 128), batch_size=32, shuffle=False)

    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    epochs = 4
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)

    best_val_f1 = -1.0
    best_state = None

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for batch in tqdm(train_loader, desc=f"Task 3 Epoch {epoch}/{epochs}"):
            optimizer.zero_grad()
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            lid_ids = batch["lid_ids"].to(device)
            labels = batch["labels"].to(device)

            loss, _ = model(input_ids=input_ids, attention_mask=attention_mask, lid_ids=lid_ids, labels=labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        val_f1, val_acc, _, _, _ = evaluate(model, val_loader, device)
        print(f"Epoch {epoch} | Train Loss: {total_loss/len(train_loader):.4f} | Val Macro-F1: {val_f1:.4f} | Val Acc: {val_acc:.4f}")
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    test_f1, test_acc, y_true, y_pred, test_records = evaluate(model, test_loader, device)
    print(f"\nTask 3 Test Macro-F1 (LID-Augmented + Normalized MuRIL): {test_f1:.4f} (Acc: {test_acc:.4f})")
    print(classification_report(y_true, y_pred, target_names=["negative", "neutral", "positive"], digits=4))

    results = {
        "task3_val_macro_f1": float(best_val_f1),
        "task3_test_macro_f1": float(test_f1),
        "task3_test_accuracy": float(test_acc)
    }
    with open(os.path.join(RESULTS_DIR, "task3_metrics.json"), "w") as f:
        json.dump(results, f, indent=2)
    with open(os.path.join(RESULTS_DIR, "task3_predictions.json"), "w", encoding="utf-8") as f:
        json.dump(test_records, f, ensure_ascii=False, indent=2)

## Run

In [ ]:
main()

Metrics are in `results/task3_metrics.json`. Next: `plots.ipynb`.